# TFEX Underlying Price Service

Learn how to fetch the **underlying instrument price** for a TFEX series — for SET50 index options and futures, the underlying instrument is the **SET50 index** itself.

## Overview

### What is the Underlying Price?

Every TFEX series (a future or an option) is derived from an **underlying instrument**. For the SET50 product family (e.g. `S50M26`, `S50M26C880`, `S50M26P880`), that underlying is the **SET50 index**.

The Underlying Price Service resolves a *series* symbol and returns the live snapshot of its *underlying* instrument:

- **Underlying symbol & type** — e.g. `SET50`, with `underlying_type` `'I'` (index)
- **Prices** — `prior` (prior close), `last`, intraday `high` / `low`
- **Change** — absolute `change` and `percent_change` from prior close, plus a `sign` indicator
- **Activity** — `total_volume` and `total_value` of the underlying
- **Market state** — `market_status` (e.g. `Open` / `Closed`) and `market_time`
- **Valuation** — `pe` (price-to-earnings) and `pbv` (price-to-book-value) of the underlying
- **As-of** — `statistics_as_of` timestamp the figures are reported as of

### Why It Matters

The underlying spot price is the anchor for derivatives pricing and analysis:

- **Basis / forward analysis** — compare the futures price against the underlying spot to measure the basis (forward premium or discount).
- **Moneyness** — for options, the spot level tells you how far in/out-of-the-money a strike is.
- **Index context** — the SET50 level, its day change, P/E and P/BV provide the market backdrop for any SET50 derivative.

### What You Get Back

`get_underlying_price(series_symbol)` returns an `UnderlyingPrice` model. Note that the returned `symbol` is the **underlying** (e.g. `SET50`), **not** the series you queried. Financial fields are `float | None` and reject `NaN`/`Infinity` at decode time to avoid silent financial-data corruption.

## Prerequisites

### Installation

You need to install the `settfex` package:

```bash
pip install settfex
```

For the pandas/CSV export cells at the end, also install the optional examples extras:

```bash
pip install "settfex[examples]"
```

### Required Imports

This notebook uses:
- `settfex` - The main library for TFEX data
- `pandas` - For tabular analysis and CSV export

In [ ]:
# Install settfex (uncomment to run)
# !pip install settfex
# !pip install "settfex[examples]"  # for the pandas/CSV export cells

In [ ]:
# Import required libraries

# For data analysis
import pandas as pd

# Import TFEX services
from settfex.services.tfex import (
    TFEXUnderlyingPriceService,
    get_series_list,
    get_underlying_price,
)

## Basic Usage

Let's resolve a single SET50 option series to its underlying instrument and inspect the snapshot.

In [ ]:
# Fetch the underlying price for S50M26C880 (a SET50 index option)
# Note: Jupyter notebooks handle async automatically in most modern versions
price = await get_underlying_price("S50M26C880")

# The returned symbol is the UNDERLYING instrument, not the series queried
print("Queried series:      S50M26C880")
print(f"Underlying symbol:   {price.symbol}")
print(f"Underlying type:     {price.underlying_type}  # 'I' = index")
print(f"Market status:       {price.market_status}")
print(f"Market time:         {price.market_time}")
print(f"Statistics as of:    {price.statistics_as_of}")

# Expected output (values vary with the market):
# Queried series:      S50M26C880
# Underlying symbol:   SET50
# Underlying type:     I  # 'I' = index
# Market status:       Open
# Market time:         2026-06-17 16:30:00+07:00
# Statistics as of:    2026-06-17 16:30:00+07:00

### Understanding the Price Fields

The underlying snapshot carries the prior close, the intraday range, the last price, and the day's change. All financial fields are `float | None`.

In [ ]:
# Display the underlying's pricing snapshot
print(f"UNDERLYING PRICE - {price.symbol}\n")
print(f"Prior Close: {price.prior}")
print(f"Last:        {price.last}")
print(f"High:        {price.high}")
print(f"Low:         {price.low}")
print(f"Change:      {price.change} ({price.percent_change}%)  sign='{price.sign}'")

# Cross-check: the reported change should equal last - prior (within rounding)
if price.last is not None and price.prior is not None:
    implied = price.last - price.prior
    print(f"\nImplied change (last - prior): {implied:+.2f}")

# Expected output (values vary with the market):
# UNDERLYING PRICE - SET50
#
# Prior Close: 925.50
# Last:        930.20
# High:        931.80
# Low:         923.10
# Change:      4.70 (0.51%)  sign='+'
#
# Implied change (last - prior): +4.70

### Activity & Valuation

Beyond price, the underlying snapshot includes the day's traded volume/value and the index-level valuation ratios (P/E and P/BV).

In [ ]:
# Display the underlying's activity and valuation ratios
print(f"ACTIVITY & VALUATION - {price.symbol}\n")
print(f"Total Volume: {price.total_volume}")
print(f"Total Value:  {price.total_value}")
print(f"P/E:          {price.pe}")
print(f"P/BV:         {price.pbv}")

# Expected output (values vary with the market):
# ACTIVITY & VALUATION - SET50
#
# Total Volume: 412345678.0
# Total Value:  18234567890.0
# P/E:          16.42
# P/BV:         1.58

## Using the Service Class

The convenience function `get_underlying_price()` wraps `TFEXUnderlyingPriceService`. Use the class directly when you want to reuse one service (and its warmed session) across many calls, or when you need the raw, unvalidated dictionary via `get_underlying_price_raw()`.

In [ ]:
# Reuse one service instance across calls
service = TFEXUnderlyingPriceService()

# Validated model
price = await service.get_underlying_price("S50M26C880")
print(f"Validated model: underlying={price.symbol}, last={price.last}")

# Raw dict (no Pydantic validation) - handy for debugging the API shape
raw = await service.get_underlying_price_raw("S50M26C880")
print(f"\nRaw response keys: {list(raw.keys())}")
print(
    f"Raw symbol={raw.get('symbol')}, last={raw.get('last')}, underlyingType={raw.get('underlyingType')}"
)

## Use Case: Spot vs Futures Basis

The **basis** is the gap between a futures price and the price of its underlying spot:

```
basis = futures_price - underlying_spot
```

A positive basis (futures above spot) is a forward **premium**; a negative basis is a forward **discount**. The basis converges toward zero as the contract approaches expiry.

Because the underlying-price endpoint resolves *any* SET50 series to the same SET50 spot, you can read the spot once and compare it against the same-expiry future. The cell below reads the spot from the underlying endpoint and the future's settlement from the Trading Statistics service, then reports the basis.

> Note: combine this with `get_trading_statistics()` (see `02_trading_statistics.ipynb`) for the future's settlement price.

In [ ]:
# Spot vs same-expiry futures forward (basis)
from settfex.services.tfex import get_trading_statistics

future_symbol = "S50M26"  # June 2026 SET50 future

# Underlying spot (SET50 index) resolved from the future series
spot = await get_underlying_price(future_symbol)

# Same-expiry future's settlement price
fut_stats = await get_trading_statistics(future_symbol)

if spot.last is not None:
    basis = fut_stats.settlement_price - spot.last
    pct = (basis / spot.last) * 100
    direction = "premium" if basis > 0 else "discount" if basis < 0 else "flat"

    print(f"BASIS ANALYSIS - {future_symbol}\n")
    print(f"Underlying ({spot.symbol}) spot last: {spot.last:.2f}")
    print(f"Future settlement price:            {fut_stats.settlement_price:.2f}")
    print(f"Days to maturity:                   {fut_stats.day_to_maturity}")
    print(f"\nBasis (future - spot): {basis:+.2f} ({pct:+.2f}%) -> forward {direction}")
else:
    print("Underlying last price unavailable (market may be closed) - basis not computable.")

# Expected output (values vary with the market):
# BASIS ANALYSIS - S50M26
#
# Underlying (SET50) spot last: 930.20
# Future settlement price:            931.50
# Days to maturity:                   12
#
# Basis (future - spot): +1.30 (+0.14%) -> forward premium

## Multiple Series Share One Underlying

A future and its options on the same product all resolve to the **same** underlying (the SET50 index). The cell below confirms that and builds a small comparison table.

In [ ]:
# Resolve several SET50 series and collect their (shared) underlying snapshot
series_symbols = ["S50M26", "S50M26C880", "S50M26P880"]

rows = []
for sym in series_symbols:
    try:
        p = await get_underlying_price(sym)
        rows.append(
            {
                "series": sym,
                "underlying": p.symbol,
                "type": p.underlying_type,
                "last": p.last,
                "change": p.change,
                "percent_change": p.percent_change,
                "pe": p.pe,
                "pbv": p.pbv,
                "market_status": p.market_status,
            }
        )
    except Exception as e:
        print(f"{sym}: Error - {e}")

df = pd.DataFrame(rows)

print("UNDERLYING SNAPSHOT BY SERIES:\n")
print(df.to_string(index=False))

# All SET50 series should resolve to one underlying instrument
underlyings = set(df["underlying"]) if not df.empty else set()
if len(underlyings) == 1:
    print(f"\n✓ All {len(df)} series share the same underlying: {next(iter(underlyings))}")

### Resolve Underlyings for Active Series

Pull a few active futures from the series-list service and resolve each to its underlying.

In [ ]:
# Resolve the underlying for active futures discovered via the series list
series_list = await get_series_list()
active_futures = [s for s in series_list.get_futures() if s.active][:5]

print(f"Resolving underlyings for {len(active_futures)} active futures...\n")
print(f"{'Series':<14} {'Underlying':<12} {'Last':<12} {'%Chg':<10} {'Status':<10}")
print("-" * 60)

for series in active_futures:
    try:
        p = await get_underlying_price(series.symbol)
        last = "n/a" if p.last is None else f"{p.last:.2f}"
        pct = "n/a" if p.percent_change is None else f"{p.percent_change:+.2f}%"
        print(f"{series.symbol:<14} {p.symbol:<12} {last:<12} {pct:<10} {p.market_status:<10}")
    except Exception as e:
        print(f"{series.symbol:<14} Error: {e}")

## Export to CSV

Snapshot the underlying for a set of series and write it to a timestamped CSV for downstream analysis (consistent with the other TFEX/SET notebooks).

In [ ]:
from datetime import datetime


async def export_underlying_snapshot(series_symbols: list[str]) -> tuple[pd.DataFrame, str]:
    """Resolve underlyings for a list of series and export to a timestamped CSV."""
    records = []
    for sym in series_symbols:
        try:
            p = await get_underlying_price(sym)
            records.append(
                {
                    "series": sym,
                    "underlying": p.symbol,
                    "underlying_type": p.underlying_type,
                    "prior": p.prior,
                    "last": p.last,
                    "high": p.high,
                    "low": p.low,
                    "change": p.change,
                    "percent_change": p.percent_change,
                    "total_volume": p.total_volume,
                    "total_value": p.total_value,
                    "pe": p.pe,
                    "pbv": p.pbv,
                    "market_status": p.market_status,
                    "market_time": p.market_time,
                    "statistics_as_of": p.statistics_as_of,
                }
            )
        except Exception as e:
            print(f"{sym}: Error - {e}")

    frame = pd.DataFrame(records)

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    filename = f"tfex_underlying_price_{timestamp}.csv"
    frame.to_csv(filename, index=False)
    return frame, filename


snapshot_df, csv_file = await export_underlying_snapshot(["S50M26", "S50M26C880", "S50M26P880"])

print(f"Underlying snapshot ({len(snapshot_df)} rows):\n")
print(snapshot_df.to_string(index=False))
print(f"\nExported to: {csv_file}")

## Error Handling

Wrap calls defensively: the request can fail, the symbol may not resolve, and a malformed payload (e.g. a `NaN` price literal) is rejected at decode time rather than silently corrupting your data.

In [ ]:
# Safe fetch with error handling
async def safe_underlying_price(symbol: str):
    """Fetch the underlying price with comprehensive error handling."""
    try:
        p = await get_underlying_price(symbol)
        print(f"✓ {symbol} -> underlying {p.symbol} (last={p.last})")
        return p
    except ConnectionError as e:
        print(f"✗ Network error for {symbol}: {e}")
        print("  Check your internet connection and try again.")
        return None
    except Exception as e:
        print(f"✗ Could not resolve underlying for {symbol}: {e}")
        print("  Verify the series exists using the series list service.")
        return None


print("Valid series:")
await safe_underlying_price("S50M26C880")

print("\nPotentially invalid series:")
await safe_underlying_price("INVALID123")

In [ ]:
# Verify the series exists before resolving its underlying
async def verify_and_resolve(symbol: str):
    """Confirm the symbol is in the TFEX series list, then resolve its underlying."""
    print(f"Verifying {symbol}...")
    series_list = await get_series_list()
    series = series_list.get_symbol(symbol)

    if not series:
        print(f"✗ Symbol {symbol} not found in the TFEX series list")
        prefix = symbol[:3]
        similar = [s.symbol for s in series_list.series if s.symbol.startswith(prefix)][:5]
        if similar:
            print(f"  Symbols starting with '{prefix}': {', '.join(similar)}")
        return None

    if not series.active:
        print(f"⚠️  {symbol} exists but is NOT ACTIVE (expiry {series.last_trading_date.date()})")

    print("✓ Symbol verified. Resolving underlying...")
    return await get_underlying_price(symbol)


price = await verify_and_resolve("S50M26C880")
if price:
    print(f"\n✓ Underlying: {price.symbol}, last={price.last}, change={price.change}")

## Next Steps

You now know how to resolve any TFEX series to its underlying instrument and read the underlying's live price, change, valuation ratios, and market status.

### Related Notebooks
- **01_series_list.ipynb** - Discover and filter available TFEX contracts
- **02_trading_statistics.ipynb** - Settlement prices, margin (IM/MM), and days to maturity (pair with this for basis analysis)

### Documentation
- [TFEX Series List Docs](../../docs/settfex/services/tfex/list.md) - Contract discovery and filtering
- [TFEX Trading Statistics Docs](../../docs/settfex/services/tfex/trading_statistics.md) - Settlement, margin, theoretical price

### Key Takeaways

1. The endpoint resolves a **series** symbol to its **underlying** instrument (SET50 index for SET50 derivatives).
2. The returned `symbol` is the **underlying** (e.g. `SET50`), not the series you queried.
3. All SET50 series (future + its options) share the **same** underlying.
4. Financial fields are `float | None` and reject `NaN`/`Infinity` at decode time.
5. **Basis** = future price - underlying spot; a positive basis is a forward premium.
6. Use `get_underlying_price_raw()` to inspect the raw API payload during debugging.

Happy trading! 📊